In [0]:
%sql
DROP TABLE IF EXISTS catalog_ventas.gold.kpi_ventas_mes;

CREATE OR REPLACE TABLE catalog_ventas.gold.kpi_ventas_mes
USING DELTA
COMMENT 'KPI Ventas mensuales'
AS
WITH base AS (

SELECT
    fa.id_venta,
    fa.cant,
    fa.total,
    fa.delivery,
    fa.vtaestado,
    f.mes
    
FROM catalog_ventas.gold.fact_ventas fa

LEFT JOIN catalog_ventas.gold.dim_fecha f
       ON fa.id_fecha = f.id_fecha

WHERE fa.vtaestado != 'anulado'

)

SELECT
    mes,
    COUNT(DISTINCT id_venta) AS total_tickets,
    SUM(total)               AS facturacion,
    SUM(cant)                AS unidades,

    ROUND(SUM(total) / 100 /  COUNT(DISTINCT id_venta),2) AS ticket_promedio,
    ROUND(SUM(cant)/ 100 / COUNT(DISTINCT id_venta),2)  AS unidades_por_ticket,

    ROUND(COUNT(DISTINCT CASE WHEN delivery = TRUE THEN id_venta END)
        *100.00 / COUNT(DISTINCT id_venta),2 )AS pct_delivery,

    CURRENT_TIMESTAMP() AS _refresh_timestamp

FROM base
GROUP BY mes 
ORDER BY mes

In [0]:
%sql
/*

Cards:

💰 Facturación total

🧾 Tickets

🍨 Unidades

🎟️ Ticket promedio

Gráficos:

Ventas por día

Ventas por hora (heatmap 🔥)

Top productos

Mix medios de pago

% delivery vs local

*/

In [0]:
%sql
select * from catalog_ventas.gold.kpi_ventas_mes

In [0]:
%sql
select vtaestado from  catalog_ventas.gold.fact_ventas 

In [0]:
%sql
select * from catalog_ventas.gold.fact_ventas limit 50